In [1]:
import os
from ebooklib import epub
import pymupdf
import re

# Directory containing the PDF files
pdf_dir = r'examples'

print(f"{'Title':<40} {'Subject':<40} {'Grade':<10}")
print(f"{'-'*40} {'-'*40} {'-'*10}")

# Iterate through the PDF files in the directory
for root, dirs, files in os.walk(pdf_dir):
    if 'CKLA' in root:
        print("Skipping CKLA")
        continue
    elif 'Final epubs' in root:
        print("Skipping Final epubs")
        continue

    for filename in files:

        # List to store image file paths
        image_files = []

        if 'CKHistory' in root:
            subject = 'Core Knowledge History and Geography'
        elif 'CKLA' in root:
            subject = 'Core Knowledge Language Arts'
        elif 'CKScience' in root:
            subject = 'Core Knowledge Science'
        else:
            subject = 'Core Knowledge'
            
        match = re.search(r'[\\\/]([1-9Kk])', root)
        if match:
            grade = match.group(1)
        else:
            grade = 'Unknown'

        new_file = True
        if filename.endswith('.pdf'):
            title = os.path.splitext(filename)[0]
            pdf_path = os.path.join(root, filename)
            pdf_document = pymupdf.open(pdf_path)
            
            print(f"{title:<40} {subject:<40} {grade:<10}")

            
            # Iterate through each page in the PDF
            for page_num in range(len(pdf_document)):
                page = pdf_document.load_page(page_num)
                pix = page.get_pixmap()
                img_path = f"tmp/{title}_page_{page_num}.png"
                pix.save(img_path)
                image_files.append(img_path)

            # Create an EPUB file
            book = epub.EpubBook()
            book.set_identifier(f'{subject}_{grade}_{title}'.strip())
            book.set_title(title)
            book.set_language('en')
            book.add_author('Core Knowledge')
            book.add_metadata(None, 'meta', '', {'name': 'calibre:series', 'content': subject})
            # Add images to the EPUB file
            for _, img_path in enumerate(image_files):
                if new_file:
                    new_file = False
                    # Set the cover image
                    book.set_cover(file_name = f'static/{img_path}', content = open(img_path, 'rb').read(), create_page=False)

                    c1 = epub.EpubHtml(title='Chapter with image', file_name='chapter_image.xhtml', lang='en')
                    chapter_html = f'''<html>
                <body>
                    '''
                    continue

                chapter_html += f'<img src="static/{img_path}"/>'

                image_content = open(img_path, 'rb').read()
                img = epub.EpubImage(uid=img_path, file_name=f'static/{img_path}', media_type='image/jpeg', content=image_content)

                # add images to the book
                book.add_item(img)

            chapter_html += '''</body>
                </html>'''
            c1.content=chapter_html

            # add chapters to the book
            book.add_item(c1)

            # Define the spine of the book
            book.spine = ['cover', 'nav', c1]

            # Write the EPUB file
            epub.write_epub(f'final_epubs/{title}.epub', book, {})

            # Clean up image files
            for img_path in image_files:
                os.remove(img_path)


Title                                    Subject                                  Grade     
---------------------------------------- ---------------------------------------- ----------
Exploring and Moving to America          Core Knowledge History and Geography     K         
Lets Explore Our World                   Core Knowledge History and Geography     K         
Mount Rushmore Presidents                Core Knowledge History and Geography     K         
Native Americans                         Core Knowledge History and Geography     K         
Skipping CKLA
Skipping CKLA
Skipping CKLA
Skipping CKLA
Skipping CKLA
Skipping CKLA
Skipping CKLA
Skipping CKLA
Skipping CKLA
Skipping CKLA
Changing Environments                    Core Knowledge Science                   K         
Computers All Around Us                  Core Knowledge Science                   K         
Needs of Plants and Animals              Core Knowledge Science                   K         
Our Five Senses        